In [2]:
# 1/12 - YOLOv8n VOC FP32. Run this setup cell before the other 11 cells.
from pathlib import Path
import torch
import tensorrt as trt
from ultralytics import YOLO, settings
import ultralytics.data.utils as data_utils

def find_project_root(start=Path.cwd()):
    for folder in (start, *start.parents):
        if (folder / 'original').is_dir() and (folder / 'PTQ').is_dir():
            return folder.resolve()
    raise FileNotFoundError('Cannot find the Dissertation project root')

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / 'PTQ' / 'datasets'
DATASETS_DIR.mkdir(exist_ok=True)
settings.update({'datasets_dir': str(DATASETS_DIR)})
data_utils.DATASETS_DIR = DATASETS_DIR  # Also update an already running notebook kernel.
DEVICE = 0
IMGSZ = 640
CALIBRATION_FRACTION = 1.0

if not torch.cuda.is_available():
    raise RuntimeError('TensorRT export requires CUDA-enabled PyTorch and an NVIDIA GPU')
print('CUDA:', torch.cuda.get_device_name(DEVICE), '| TensorRT:', trt.__version__)

def export_engine(relative_weights, precision, data=None):
    if precision not in {'fp32', 'fp16', 'int8'}:
        raise ValueError(f'Unsupported precision: {precision}')
    source = PROJECT_ROOT / relative_weights
    if not source.is_file():
        raise FileNotFoundError(source)
    if precision == 'int8' and data is None:
        raise ValueError('INT8 export requires a matching calibration dataset')
    model = YOLO(str(source))
    model.model.pt_path = str(source.with_name(f'{source.stem}_{precision}.pt'))
    options = dict(format='engine', device=DEVICE, imgsz=IMGSZ, batch=1,
                   dynamic=False, simplify=True, fraction=CALIBRATION_FRACTION,
                   quantize={'fp32': 32, 'fp16': 16, 'int8': 8}[precision])
    if data is not None:
        options['data'] = data
    output = Path(model.export(**options)).resolve()
    if output.suffix != '.engine' or not output.is_file():
        raise RuntimeError(f'TensorRT engine was not created: {output}')
    print(f'{relative_weights} [{precision.upper()}] -> {output} ({output.stat().st_size / 2**20:.2f} MiB)')
    return output

#export_engine('original/yolov8n_voc/voc_8n_best.pt', 'fp32')

CUDA: NVIDIA RTX A2000 | TensorRT: 11.3.0.99


In [ ]:
# 2/12 - YOLOv8n VOC FP16 TensorRT
export_engine('original/yolov8n_voc/voc_8n_best.pt', 'fp16')

In [ ]:
# 3/12 - YOLOv8n VOC INT8 TensorRT
export_engine('original/yolov8n_voc/voc_8n_best.pt', 'int8', data='VOC.yaml')

In [ ]:
# 4/12 - YOLOv8l VOC FP32 TensorRT
export_engine('original/yolo8l_voc/voc_8l_best.pt', 'fp32')

In [ ]:
# 5/12 - YOLOv8l VOC FP16 TensorRT
export_engine('original/yolo8l_voc/voc_8l_best.pt', 'fp16')

In [ ]:
# 6/12 - YOLOv8l VOC INT8 TensorRT
export_engine('original/yolo8l_voc/voc_8l_best.pt', 'int8', data='VOC.yaml')

In [ ]:
# 7/12 - YOLOv8n KITTI FP32 TensorRT
export_engine('original/yolov8n_kitti/8n_float32.pt', 'fp32')

In [ ]:
# 8/12 - YOLOv8n KITTI FP16 TensorRT
export_engine('original/yolov8n_kitti/8n_float32.pt', 'fp16')

In [3]:
# 9/12 - YOLOv8n KITTI INT8 TensorRT
export_engine('original/yolov8n_kitti/8n_float32.pt', 'int8', data='kitti.yaml')

Ultralytics 8.4.163  Python-3.14.7 torch-2.14.0+cu130 CUDA:0 (NVIDIA RTX A2000, 6138MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_int8.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (0.0 MB)

ONNX: starting export with onnx 1.23.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success  1.4s, saved as 'C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_int8.onnx' (11.7 MB)
TensorRT: collecting INT8 calibration images from 'data=kitti.yaml'
val: Fast image access  (ping: 0.10.0 ms, read: 326.7111.9 MB/s, size: 49.3 KB)
val: Scanning C:\Users\dxu1\Desktop\Dissertation\PTQ\datasets\kitti\labels\val.cache... 1496 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1496/1496 448.2Mit/s 0.0s

TensorRT: starting export with TensorRT 11.3.0.99...
TensorRT: quantizing ONNX to INT8 with ModelOpt using 

WindowsPath('C:/Users/dxu1/Desktop/Dissertation/original/yolov8n_kitti/8n_float32_int8.engine')

In [ ]:
# 10/12 - YOLOv8l KITTI FP32 TensorRT
export_engine('original/yolov8l_kitti/8l_float32.pt', 'fp32')

In [ ]:
# 11/12 - YOLOv8l KITTI FP16 TensorRT
export_engine('original/yolov8l_kitti/8l_float32.pt', 'fp16')

In [1]:
# 12/12 - YOLOv8l KITTI INT8 TensorRT
export_engine('original/yolov8l_kitti/8l_float32.pt', 'int8', data='kitti.yaml')

NameError: name 'export_engine' is not defined